In [5]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam

# --- CONFIG ---
IMG_SIZE = 128
BATCH_SIZE = 32
LATENT_DIM = 128
EPOCHS = 50
DATA_DIR = r"C:\Users\User\Downloads\Skin-disease-dataset\mixed"  # <-- Change this

# --- IMAGE DATA AUGMENTATION ---
datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
    vertical_flip=True
)

train_gen = datagen.flow_from_directory(
    directory=os.path.dirname(DATA_DIR),  # parent folder
    classes=[os.path.basename(DATA_DIR)],  # target folder only
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode=None,
    shuffle=True
)

# --- ENCODER ---
def build_encoder():
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = layers.Conv2D(64, 4, strides=2, padding='same', activation='relu')(inputs)
    x = layers.Conv2D(128, 4, strides=2, padding='same', activation='relu')(x)
    x = layers.Conv2D(256, 4, strides=2, padding='same', activation='relu')(x)
    x = layers.Flatten()(x)
    z = layers.Dense(LATENT_DIM)(x)
    return Model(inputs, z, name="Encoder")

# --- DECODER ---
def build_decoder():
    inputs = layers.Input(shape=(LATENT_DIM,))
    x = layers.Dense((IMG_SIZE//8)*(IMG_SIZE//8)*256)(inputs)
    x = layers.Reshape((IMG_SIZE//8, IMG_SIZE//8, 256))(x)
    x = layers.Conv2DTranspose(128, 4, strides=2, padding='same', activation='relu')(x)
    x = layers.Conv2DTranspose(64, 4, strides=2, padding='same', activation='relu')(x)
    x = layers.Conv2DTranspose(3, 4, strides=2, padding='same', activation='sigmoid')(x)
    return Model(inputs, x, name="Decoder")

# --- DISCRIMINATOR ---
def build_discriminator():
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = layers.Conv2D(64, 4, strides=2, padding='same', activation='leaky_relu')(inputs)
    x = layers.Conv2D(128, 4, strides=2, padding='same', activation='leaky_relu')(x)
    x = layers.Conv2D(256, 4, strides=2, padding='same', activation='leaky_relu')(x)
    features = layers.GlobalAveragePooling2D()(x)
    validity = layers.Dense(1, activation='sigmoid')(features)
    return Model(inputs, [validity, features], name="Discriminator")

# --- MODEL INITIALIZATION ---
encoder = build_encoder()
decoder = build_decoder()
discriminator = build_discriminator()

# Combined Generator (G = GE + GD)
input_img = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
z = encoder(input_img)
x_hat = decoder(z)
generator = Model(input_img, x_hat, name="Generator")

# Encoder for generated image
z_hat = encoder(x_hat)

# --- LOSSES ---
bce = tf.keras.losses.BinaryCrossentropy()

def adversarial_loss(real_feat, fake_feat):
    return tf.reduce_mean(tf.square(real_feat - fake_feat))

def contextual_loss(real_img, recon_img):
    return tf.reduce_mean(tf.abs(real_img - recon_img))

def encoder_loss(z, z_hat):
    return tf.reduce_mean(tf.square(z - z_hat))

# --- OPTIMIZERS ---
g_opt = Adam(1e-4)
d_opt = Adam(1e-4)

# --- TRAINING LOOP ---
@tf.function
def train_step(images):
    with tf.GradientTape(persistent=True) as tape:
        z = encoder(images)
        x_hat = decoder(z)
        z_hat = encoder(x_hat)

        # Discriminator outputs
        real_pred, real_feat = discriminator(images)
        fake_pred, fake_feat = discriminator(x_hat)

        # Losses
        d_loss_real = bce(tf.ones_like(real_pred), real_pred)
        d_loss_fake = bce(tf.zeros_like(fake_pred), fake_pred)
        d_loss = d_loss_real + d_loss_fake

        adv_loss = adversarial_loss(real_feat, fake_feat)
        con_loss = contextual_loss(images, x_hat)
        enc_loss = encoder_loss(z, z_hat)
        g_loss = adv_loss + con_loss + enc_loss

    # Backpropagation
    d_grads = tape.gradient(d_loss, discriminator.trainable_variables)
    g_grads = tape.gradient(g_loss, generator.trainable_variables + encoder.trainable_variables)

    d_opt.apply_gradients(zip(d_grads, discriminator.trainable_variables))
    g_opt.apply_gradients(zip(g_grads, generator.trainable_variables + encoder.trainable_variables))

    return d_loss, g_loss

# --- TRAINING ---
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    for step, batch in enumerate(train_gen):
        d_loss, g_loss = train_step(batch)
        if step % 10 == 0:
            print(f"Step {step}: D_loss={d_loss.numpy():.4f}, G_loss={g_loss.numpy():.4f}")
        if step >= len(train_gen):
            break

# --- SAVE MODELS ---
encoder.save("ganomaly_encoder.h5")
decoder.save("ganomaly_decoder.h5")
discriminator.save("ganomaly_discriminator.h5")
generator.save("ganomaly_generator.h5")


Found 41096 images belonging to 1 classes.

Epoch 1/50
Step 0: D_loss=1.3793, G_loss=0.2323
Step 10: D_loss=1.2681, G_loss=0.2521
Step 20: D_loss=1.0148, G_loss=0.2432
Step 30: D_loss=0.7065, G_loss=0.2649
Step 40: D_loss=0.5951, G_loss=0.2788
Step 50: D_loss=0.4471, G_loss=0.2910
Step 60: D_loss=0.4322, G_loss=0.3545
Step 70: D_loss=3.0074, G_loss=0.2753
Step 80: D_loss=0.9220, G_loss=0.2834
Step 90: D_loss=1.0282, G_loss=0.2607
Step 100: D_loss=1.6176, G_loss=0.2348
Step 110: D_loss=1.3821, G_loss=0.2210
Step 120: D_loss=1.8751, G_loss=0.2038
Step 130: D_loss=1.5851, G_loss=0.2148
Step 140: D_loss=1.4994, G_loss=0.2026
Step 150: D_loss=1.4394, G_loss=0.2492
Step 160: D_loss=1.3406, G_loss=0.2065
Step 170: D_loss=1.3176, G_loss=0.2050
Step 180: D_loss=1.3582, G_loss=0.2236
Step 190: D_loss=1.3348, G_loss=0.2311
Step 200: D_loss=1.3105, G_loss=0.2053
Step 210: D_loss=1.2927, G_loss=0.2244
Step 220: D_loss=1.2829, G_loss=0.2171
Step 230: D_loss=1.2553, G_loss=0.2133
Step 240: D_loss=1.2

KeyboardInterrupt: 

In [12]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam

# --- CONFIG ---
IMG_SIZE = 224
BATCH_SIZE = 16
LATENT_DIM = 128
EPOCHS = 30
DATA_DIR = r"C:\Users\User\Downloads\Skin-disease-dataset\mixed"  # <-- Change this
PRETRAINED_ENCODER_PATH = "four_skin_disease_detection.h5"  # Path to your pretrained encoder model

# --- IMAGE DATA AUGMENTATION ---
datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='nearest'
)

train_gen = datagen.flow_from_directory(
    directory=os.path.dirname(DATA_DIR),
    classes=[os.path.basename(DATA_DIR)],
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode=None,
    shuffle=True
)

# --- LOAD PRETRAINED ENCODER AND ADD LAYERS ---
base_encoder = tf.keras.models.load_model(PRETRAINED_ENCODER_PATH)
base_encoder.trainable = True  # Set False if you want to freeze pretrained layers initially

# Suppose base_encoder outputs a vector, add layers on top for GANomaly
x = base_encoder.output
x = layers.Dense(256, activation='relu')(x)
z = layers.Dense(LATENT_DIM, name="latent_vector")(x)
encoder = Model(base_encoder.input, z, name="Encoder")

# --- DECODER ---
def build_decoder():
    inputs = layers.Input(shape=(LATENT_DIM,))
    x = layers.Dense((IMG_SIZE // 8) * (IMG_SIZE // 8) * 256, activation='relu')(inputs)
    x = layers.Reshape((IMG_SIZE // 8, IMG_SIZE // 8, 256))(x)  # (28, 28, 256)

    x = layers.Conv2DTranspose(128, 4, strides=2, padding='same', activation='relu')(x)
    x = layers.Conv2DTranspose(64, 4, strides=2, padding='same', activation='relu')(x)
    x = layers.Conv2DTranspose(3, 4, strides=2, padding='same', activation='sigmoid')(x)
    return Model(inputs, x, name="Decoder")

decoder = build_decoder()

# --- DISCRIMINATOR ---
def build_discriminator():
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = layers.Conv2D(64, 4, strides=2, padding='same')(inputs)
    x = layers.LeakyReLU(alpha=0.2)(x)
    x = layers.Conv2D(128, 4, strides=2, padding='same')(x)
    x = layers.LeakyReLU(alpha=0.2)(x)
    x = layers.Conv2D(256, 4, strides=2, padding='same')(x)
    x = layers.LeakyReLU(alpha=0.2)(x)
    features = layers.GlobalAveragePooling2D()(x)
    validity = layers.Dense(1, activation='sigmoid')(features)
    return Model(inputs, [validity, features], name="Discriminator")

discriminator = build_discriminator()

# --- GENERATOR (Encoder + Decoder) ---
input_img = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
z = encoder(input_img)
x_hat = decoder(z)
generator = Model(input_img, x_hat, name="Generator")

# Encoder for generated image
z_hat = encoder(x_hat)

# --- LOSSES ---
bce = tf.keras.losses.BinaryCrossentropy()

def adversarial_loss(real_feat, fake_feat):
    return tf.reduce_mean(tf.square(real_feat - fake_feat))

def contextual_loss(real_img, recon_img):
    return tf.reduce_mean(tf.abs(real_img - recon_img))

def encoder_loss(z, z_hat):
    return tf.reduce_mean(tf.square(z - z_hat))

# --- OPTIMIZERS ---
g_opt = Adam(1e-4)
d_opt = Adam(1e-4)

# --- TRAINING STEP ---
@tf.function
def train_step(images):
    with tf.GradientTape(persistent=True) as tape:
        z = encoder(images)
        x_hat = decoder(z)
        z_hat = encoder(x_hat)

        real_pred, real_feat = discriminator(images)
        fake_pred, fake_feat = discriminator(x_hat)

        d_loss_real = bce(tf.ones_like(real_pred), real_pred)
        d_loss_fake = bce(tf.zeros_like(fake_pred), fake_pred)
        d_loss = d_loss_real + d_loss_fake

        adv_loss = adversarial_loss(real_feat, fake_feat)
        con_loss = contextual_loss(images, x_hat)
        enc_loss = encoder_loss(z, z_hat)
        g_loss = adv_loss + con_loss + enc_loss

    d_grads = tape.gradient(d_loss, discriminator.trainable_variables)
    g_grads = tape.gradient(g_loss, generator.trainable_variables)


    d_opt.apply_gradients(zip(d_grads, discriminator.trainable_variables))
    g_opt.apply_gradients(zip(g_grads, generator.trainable_variables + encoder.trainable_variables))

    return d_loss, g_loss

# --- TRAINING LOOP ---
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    for step, batch in enumerate(train_gen):
        d_loss, g_loss = train_step(batch)
        if step % 10 == 0:
            print(f"Step {step}: D_loss={d_loss.numpy():.4f}, G_loss={g_loss.numpy():.4f}")
        if step >= len(train_gen):
            break

# --- SAVE MODELS ---
encoder.save("ganomaly_encoder_extended.h5")
decoder.save("ganomaly_decoder_extended.h5")
discriminator.save("ganomaly_discriminator_extended.h5")
generator.save("ganomaly_generator_extended.h5")


Found 41096 images belonging to 1 classes.

Epoch 1/30
Step 0: D_loss=1.3887, G_loss=0.3024
Step 10: D_loss=1.2413, G_loss=0.2699
Step 20: D_loss=0.9664, G_loss=0.2227
Step 30: D_loss=0.6668, G_loss=0.2765
Step 40: D_loss=0.4794, G_loss=0.3071
Step 50: D_loss=0.6161, G_loss=0.3057
Step 60: D_loss=0.1988, G_loss=0.3779
Step 70: D_loss=1.5823, G_loss=0.3043
Step 80: D_loss=2.7563, G_loss=0.2795
Step 90: D_loss=1.5466, G_loss=0.2013
Step 100: D_loss=1.3048, G_loss=0.2839
Step 110: D_loss=1.0919, G_loss=0.2710
Step 120: D_loss=1.2797, G_loss=0.2379
Step 130: D_loss=1.2844, G_loss=0.2461
Step 140: D_loss=1.1324, G_loss=0.2382
Step 150: D_loss=1.2554, G_loss=0.2118
Step 160: D_loss=1.0928, G_loss=0.2330
Step 170: D_loss=0.9643, G_loss=0.2327
Step 180: D_loss=1.0875, G_loss=0.2422
Step 190: D_loss=0.9921, G_loss=0.2450
Step 200: D_loss=1.0049, G_loss=0.2575
Step 210: D_loss=1.1385, G_loss=0.2319
Step 220: D_loss=0.9495, G_loss=0.2488
Step 230: D_loss=0.9803, G_loss=0.2506
Step 240: D_loss=0.8

MemoryError: Unable to allocate 9.19 MiB for an array with shape (16, 224, 224, 3) and data type float32